In [1]:
import sys, os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim


from collections import deque

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, PROJECT_ROOT)
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
os.makedirs(DATA_DIR, exist_ok=True)
from utils.hospital_env import HospitalEnv

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


In [2]:
def generate_data(n, seed=42):
    np.random.seed(seed)
    DAY = 8 * 60

    data = pd.DataFrame({
        "arrival_time": np.random.randint(0, DAY, n),
        "slot_time": np.random.randint(0, DAY, n),
        "priority": np.random.choice([0,1], n, p=[0.8,0.2]),
        "no_show_prob": np.random.uniform(0.05, 0.4, n),
    })
    data["show"] = (np.random.rand(n) > data["no_show_prob"]).astype(int)
    return data


In [3]:
def normalize(state):
    a, s, p, n, icu = state
    return np.array([
        a / 480,
        s / 480,
        p / 2.0,
        n,
        icu
    ], dtype=np.float32)


In [4]:
class DQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(5, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        return self.net(x)


In [5]:
def train_ddqn(data, episodes=50):

    

    policy = DQN()
    target = DQN()
    target.load_state_dict(policy.state_dict())
    target.eval()

    optimizer = optim.Adam(policy.parameters(), lr=1e-3)
    memory = deque(maxlen=5000)

    gamma = 0.99
    epsilon = 1.0
    epsilon_min = 0.05
    epsilon_decay = 0.995
    batch_size = 64
    target_update = 5

    for ep in range(episodes):
        env = HospitalEnv(data)
        state = normalize(env.reset()).astype(np.float32)
        done = False

        while not done:

            # -------- Epsilon-Greedy Action --------
            if random.random() < epsilon:
                action = random.randint(0, 1)
            else:
                with torch.no_grad():
                    state_tensor = torch.from_numpy(state).float().unsqueeze(0)
                    q_vals = policy(state_tensor)
                    action = torch.argmax(q_vals, dim=1).item()

            # -------- Environment Step --------
            next_state_raw, reward, done = env.step(action)

            if not done:
                next_state = normalize(next_state_raw).astype(np.float32)
            else:
                next_state = np.zeros(5, dtype=np.float32)

            # -------- Store Transition --------
            memory.append((state, action, reward, next_state, done))
            state = next_state

            # -------- Training Step --------
            if len(memory) >= batch_size:

                batch = random.sample(memory, batch_size)
                s, a, r, ns, d = zip(*batch)

                s  = torch.from_numpy(np.stack(s)).float()
                ns = torch.from_numpy(np.stack(ns)).float()
                a  = torch.tensor(a, dtype=torch.long)
                r  = torch.tensor(r, dtype=torch.float32)
                d  = torch.tensor(d, dtype=torch.float32)

                # Current Q-values
                q_pred = policy(s).gather(1, a.unsqueeze(1)).squeeze()

                # -------- Double DQN Target --------
                with torch.no_grad():
                    next_actions = policy(ns).argmax(1)
                    q_next = target(ns).gather(
                        1, next_actions.unsqueeze(1)
                    ).squeeze()

                    q_target = r + gamma * q_next * (1 - d)

                loss = nn.MSELoss()(q_pred, q_target)

                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
                optimizer.step()

        # -------- Target Network Update --------
        if (ep + 1) % target_update == 0:
            target.load_state_dict(policy.state_dict())

        # -------- Epsilon Decay --------
        epsilon = max(epsilon_min, epsilon * epsilon_decay)

        print(f"Episode {ep+1}/{episodes} completed")

    # -------- Save Model --------
    os.makedirs("models", exist_ok=True)
    torch.save(policy.state_dict(), "models/ddqn_model.pt")

    # -------- Final Evaluation --------
    return evaluate(policy, HospitalEnv(data))


In [6]:
def evaluate(model, env):
    state = normalize(env.reset()).astype(np.float32)
    done = False
    total_reward = 0

    model.eval()  # switch to evaluation mode

    while not done:
        with torch.no_grad():
            state_tensor = torch.from_numpy(state).float().unsqueeze(0)
            action = torch.argmax(model(state_tensor), dim=1).item()

        next_state, reward, done = env.step(action)
        total_reward += reward

        if not done:
            state = normalize(next_state).astype(np.float32)

    return total_reward


In [ ]:
sizes = [500, 2000, 5000,10000]
results = {}

for size in sizes:
    print(f"\nTraining on dataset size: {size}")
    data = generate_data(size)
    data.to_csv(os.path.join(DATA_DIR, f"synthetic_hospital_data_{size}.csv"), index=False)
    reward = train_ddqn(data)
    results[size] = reward
    print(f"Evaluation Reward: {reward}")



Training on dataset size: 500
Episode 1/50 completed
Episode 2/50 completed
Episode 3/50 completed
Episode 4/50 completed
Episode 5/50 completed
Episode 6/50 completed
Episode 7/50 completed
Episode 8/50 completed
Episode 9/50 completed
Episode 10/50 completed
Episode 11/50 completed
Episode 12/50 completed
Episode 13/50 completed
Episode 14/50 completed
Episode 15/50 completed
Episode 16/50 completed
Episode 17/50 completed
Episode 18/50 completed
Episode 19/50 completed
Episode 20/50 completed
Episode 21/50 completed
Episode 22/50 completed
Episode 23/50 completed
Episode 24/50 completed
Episode 25/50 completed
Episode 26/50 completed
Episode 27/50 completed
Episode 28/50 completed
Episode 29/50 completed
Episode 30/50 completed
Episode 31/50 completed
Episode 32/50 completed
Episode 33/50 completed
Episode 34/50 completed
Episode 35/50 completed
Episode 36/50 completed
Episode 37/50 completed
Episode 38/50 completed
Episode 39/50 completed
Episode 40/50 completed
Episode 41/50 comp

In [ ]:
plt.plot(results.keys(), results.values(), marker='o')
plt.xlabel("Dataset Size")
plt.ylabel("Total Reward")
plt.title("Phase 9: Dataset Scaling & Generalization")
plt.show()
